In [ ]:
# =====================================================
# STEP 1: CONNECT GOOGLE DRIVE
# =====================================================

# Google Colab runs on a temporary computer.
# If we don’t connect Google Drive, all files will be lost.
from google.colab import drive

# This line connects your Drive to Colab
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =====================================================
# STEP 2: IMPORT REQUIRED LIBRARIES
# =====================================================
# These tools help us:
# - Read image files
# - Work with numbers
# - Build and train a learning model

import os
import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten


In [ ]:
# =====================================================
# STEP 3: SET DATASET PATH
# =====================================================
# This is the folder that contains UTKFace images.
# Each file name contains age and gender information.

DATASET_PATH = "/content/drive/MyDrive/UTKFace"

# Get list of all image files
files = os.listdir(DATASET_PATH)

# Shuffle files so learning is random
np.random.shuffle(files)


In [ ]:
# =====================================================
# STEP 4: CREATE GENDER DATA GENERATOR
# =====================================================
# This function loads ONLY a small number of images
# at a time (called a batch).
#
# This prevents RAM from getting full and crashing.

def gender_generator(file_list, batch_size=32):

    images = []
    genders = []

    while True:  # Runs continuously during training
        for file in file_list:

            try:
                # Example filename: 25_0_1_201701.jpg
                # Second value = gender
                age, gender, _ = file.split("_", 2)

                gender = int(gender)  # 0 = Male, 1 = Female

                # Read image
                img = cv2.imread(os.path.join(DATASET_PATH, file))
                if img is None:
                    continue

                # Resize image to reduce memory usage
                img = cv2.resize(img, (128, 128))

                # Normalize pixel values (0–255 → 0–1)
                img = img / 255.0

                # Store image and label
                images.append(img)
                genders.append(gender)

                # When batch is full, send to model
                if len(images) == batch_size:
                    yield np.array(images, dtype="float32"), np.array(genders)
                    images, genders = [], []

            except:
                continue


In [ ]:
# =====================================================
# STEP 5: BUILD GENDER MODEL
# =====================================================
# This model learns facial patterns
# and predicts whether the face is Male or Female.

gender_model = Sequential([

    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')  # Male / Female
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# =====================================================
# STEP 6: COMPILE AND TRAIN GENDER MODEL
# =====================================================
# This step tells the model:
# - How to learn
# - How to measure mistakes

gender_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

BATCH_SIZE = 32
STEPS = len(files) // BATCH_SIZE

gender_model.fit(
    gender_generator(files, BATCH_SIZE),
    steps_per_epoch=STEPS,
    epochs=5
)


Epoch 1/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 316s 421ms/step - accuracy: 0.7658 - loss: 0.4925
Epoch 2/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 119s 160ms/step - accuracy: 0.8657 - loss: 0.3061
Epoch 3/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 119s 161ms/step - accuracy: 0.8875 - loss: 0.2642
Epoch 4/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 118s 160ms/step - accuracy: 0.8979 - loss: 0.2350
Epoch 5/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 118s 160ms/step - accuracy: 0.9124 - loss: 0.2097


In [ ]:
# =====================================================
# STEP 7: SAVE GENDER MODEL
# =====================================================
# Save the trained gender model for later use.

gender_model.save("/content/drive/MyDrive/gender_model.h5")


In [ ]:
# =====================================================
# STEP 8: IMPORT LIBRARIES AGAIN
# =====================================================
# After restarting runtime, we must import libraries again.

import os
import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from tensorflow.keras.utils import to_categorical


In [ ]:
# =====================================================
# STEP 9: DEFINE AGE GROUPS
# =====================================================
# Instead of predicting exact age,
# we predict age ranges to make learning easier.

age_groups = [
    (0, 2),
    (4, 6),
    (8, 12),
    (15, 20),
    (25, 32),
    (38, 43),
    (48, 100)
]

def get_age_group(age):
    for i, (low, high) in enumerate(age_groups):
        if low <= age <= high:
            return i
    return None


In [ ]:
# =====================================================
# STEP 10: CREATE AGE DATA GENERATOR
# =====================================================
# This generator loads images in small batches
# and provides age group labels.

def age_generator(file_list, batch_size=32):

    images = []
    ages = []

    while True:
        for file in file_list:

            try:
                age, gender, _ = file.split("_", 2)
                age = int(age)

                age_group = get_age_group(age)
                if age_group is None:
                    continue

                img = cv2.imread(os.path.join(DATASET_PATH, file))
                if img is None:
                    continue

                img = cv2.resize(img, (128,128))
                img = img / 255.0

                images.append(img)
                ages.append(age_group)

                if len(images) == batch_size:
                    yield np.array(images, dtype="float32"), to_categorical(ages, 7)
                    images, ages = [], []

            except:
                continue


In [ ]:
# =====================================================
# STEP 11: BUILD AGE MODEL
# =====================================================
# This model learns facial features
# and predicts the age group.

age_model = Sequential([

    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(7, activation='softmax')  # Age groups
])


In [ ]:
# =====================================================
# STEP 12: COMPILE AND TRAIN AGE MODEL
# =====================================================

age_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

age_model.fit(
    age_generator(files, BATCH_SIZE),
    steps_per_epoch=STEPS,
    epochs=10
)


Epoch 1/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 123s 161ms/step - accuracy: 0.5203 - loss: 1.4897
Epoch 2/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 118s 159ms/step - accuracy: 0.6820 - loss: 0.9064
Epoch 3/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 116s 157ms/step - accuracy: 0.7150 - loss: 0.7785
Epoch 4/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 117s 158ms/step - accuracy: 0.7445 - loss: 0.6889
Epoch 5/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 117s 158ms/step - accuracy: 0.7715 - loss: 0.6054
Epoch 6/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 117s 157ms/step - accuracy: 0.7970 - loss: 0.5377
Epoch 7/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 116s 157ms/step - accuracy: 0.8243 - loss: 0.4670
Epoch 8/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 116s 157ms/step - accuracy: 0.8437 - loss: 0.3979
Epoch 9/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 117s 158ms/step - accuracy: 0.8652 - loss: 0.3558
Epoch 10/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 117s 158ms/step - accuracy: 0.8760 - loss: 0.3193


In [ ]:
# =====================================================
# STEP 13: SAVE AGE MODEL
# =====================================================
# Save trained age model.

age_model.save("/content/drive/MyDrive/age_model.h5")
